## 05b - XDec Stage 0: Full k-Range Stability Sweep (Partitioned)

### Aim
* Run XDec Stage 0's full default stability sweep (`k=3..12`) on the whole 182-sample TCGA-PAAD cohort, using the real 262-gene informative set already derived from the `GSE111672`/Zenodo `CAF_integrated.rds`-based reference (notebook 05's follow-up pilot, not the earlier placeholder). Partitioned by k (one subprocess call per k, each checkpointed to disk) rather than one blocking `--k-range 3,4,...,12` call, per notebook 04's resource-management plan - the k=3,4 pilot showed per-k cost is NOT constant (k=4 took ~4.7x longer than k=3 on the placeholder gene list), so the full sweep's total time is unknown and must be measured empirically, one k at a time, not assumed.

### Input
* `data/processed/TCGA-PAAD_counts_raw.tsv` (raw counts, notebook 02).
* `data/deconvolution/xdec_caf_pilot/selected_probes.txt` - the 262 real informative genes already selected via one-vs-rest t-tests against the CAF-cluster reference (prior run in this conversation; re-used here via `--probes` so every k in this sweep is tested on the exact same gene set - skips re-running gene selection for each partition).

### Output
* `data/deconvolution/xdec_full_sweep/k_XX/stability_scores.tsv` - one-row stability result per k, written as each partition completes (resumable - a crash/interrupt partway through does not lose completed k's).
* `data/deconvolution/xdec_full_sweep/stability_scores.tsv` and `optimal_k.txt` - the combined result across all k, and the per-k elapsed time log (`timing_log.tsv`) giving the first real empirical growth curve across the full k=3..12 range (the pilot only had 2 points).

### Steps
1. For each k in `3:12`, skip if `data/deconvolution/xdec_full_sweep/k_XX/stability_scores.tsv` already exists (idempotent/resumable).
2. Otherwise run `xdec.py stage0 --probes selected_probes.txt --k-range <k>` as its own subprocess, timed, writing to its own `k_XX/` subdirectory.
3. Log each k's elapsed time to `timing_log.tsv` immediately after it completes (not batched at the end), so progress is visible even if the whole sweep is still running or gets interrupted.
4. Aggregate all completed per-k `stability_scores.tsv` files into one combined table and write `optimal_k.txt` (highest stability across the whole sweep).

### Result
* Not yet run - populated once this notebook is executed. Expect this to take a while (the k=3,4 pilot alone took ~71 min on a placeholder gene list; the real 262-gene list ran k=3,4 in ~12 min, but k=5..12 are unmeasured and may not follow the same trend).

### Conclusion
* Not yet run.

### Set working directory to the project root

In [ ]:
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

### Partitioned sweep: one `xdec.py stage0` subprocess per k, checkpointed

Each k is a fully independent call (same `--probes` list, same `--bulk`, different `--k-range`). Skips any k whose output directory already has a `stability_scores.tsv` - safe to re-run this cell after an interruption without redoing completed k's.

In [ ]:
py_bin      <- path.expand("~/.pixi/envs/python/bin/python")
xdec_script <- "tools/XDec_SM/python/xdec.py"
bulk_path   <- "data/processed/TCGA-PAAD_counts_raw.tsv"
probes_path <- "data/deconvolution/xdec_caf_pilot/selected_probes.txt"
sweep_dir   <- "data/deconvolution/xdec_full_sweep"
dir.create(sweep_dir, recursive = TRUE, showWarnings = FALSE)

timing_log_path <- file.path(sweep_dir, "timing_log.tsv")
if (!file.exists(timing_log_path)) {
  writeLines("k\telapsed_sec\tstability_score", timing_log_path)
}

for (k in 3:12) {
  k_dir      <- file.path(sweep_dir, sprintf("k_%02d", k))
  k_stab_tsv <- file.path(k_dir, "stability_scores.tsv")

  if (file.exists(k_stab_tsv)) {
    message(sprintf("k=%d already done - skipping.", k))
    next
  }

  dir.create(k_dir, recursive = TRUE, showWarnings = FALSE)
  args_k <- c(
    xdec_script, "stage0",
    "--bulk", bulk_path,
    "--probes", probes_path,
    "--k-range", as.character(k),
    "--outdir", k_dir
  )

  message(sprintf("Running k=%d ...", k))
  t0 <- Sys.time()
  log_k <- system2(py_bin, args_k, stdout = TRUE, stderr = TRUE)
  t1 <- Sys.time()
  elapsed_k <- as.numeric(difftime(t1, t0, units = "secs"))

  if (!file.exists(k_stab_tsv)) {
    cat(log_k, sep = "\n")
    stop(sprintf("k=%d did not produce stability_scores.tsv - stopping sweep here so completed k's are not lost. See log above.", k))
  }

  stab_k <- read.delim(k_stab_tsv)
  score_k <- stab_k$stability_score[stab_k$k == k]

  cat(sprintf("%d\t%.1f\t%.4f\n", k, elapsed_k, score_k),
      file = timing_log_path, append = TRUE)
  message(sprintf("k=%d done: %.1f sec, stability=%.4f", k, elapsed_k, score_k))
}

read.delim(timing_log_path)

### Aggregate all completed k's into the combined result

In [ ]:
k_dirs <- Sys.glob(file.path(sweep_dir, "k_*", "stability_scores.tsv"))
stopifnot(length(k_dirs) > 0)

all_stab <- do.call(rbind, lapply(k_dirs, read.delim))
all_stab <- all_stab[order(all_stab$k), ]

write.table(all_stab, file.path(sweep_dir, "stability_scores.tsv"),
            sep = "\t", quote = FALSE, row.names = FALSE)

optimal_k <- all_stab$k[which.max(all_stab$stability_score)]
writeLines(as.character(optimal_k), file.path(sweep_dir, "optimal_k.txt"))

cat(sprintf("Recommended k = %d (stability = %.4f)\n",
            optimal_k, max(all_stab$stability_score)))
all_stab